# Linear Becomes Experts

**An executed, reproducible linear → sparse MoE assignment**

Train a literal affine classifier on handwritten digits, copy its learned
weights into independent experts, then show that learning continues.
This notebook contains every implementation and runs without repository files.
**Execution status: prepared. CI will generate and verify fresh Jupyter outputs.**

**Run all** in Jupyter or Colab with a CPU. NumPy, scikit-learn, Matplotlib and
threadpoolctl are needed; they are usually already installed in Colab.
The digits data ships with scikit-learn. No GPU, API key or dataset download.

**Protocol:** 5 initialization seeds × 3 stratified
splits = 15 paired cases. All final checkpoints are fixed in advance.
Five branches compare equal steps, forward projection budgets, training from
scratch and removal of the router balancing loss. Variation across overlapping
splits is descriptive; it is not a statistical significance claim.


## 1. Models and sparse dispatch

The starting classifier has no hidden layer: $z(x)=xW+b$.
Each copied expert starts with $W_e=W$ and $b_e=b$.
With normalized selected gates, $\sum_{e\in S(x)}g_e(x)=1$, so
$z_{MoE}(x)=\sum_{e\in S(x)}g_e(x)(xW_e+b_e)=z(x)$ at conversion.

Top-2 routing lets task gradients flow through selected softmax weights.
Hard expert selection is piecewise constant and is not differentiated.
The code evaluates an expert only on assigned input rows. All experts have
independent storage; no inputs are dropped. The scratch baseline uses the
same architecture with independently initialized untrained expert weights.


In [ ]:
"""Train a literal linear classifier, upcycle to top-2 MoE, continue both.

NumPy implements the model, exact gradients, and Adam. scikit-learn is used
only for its bundled digits dataset and stratified splitting, never training.
"""
import argparse
import copy
import csv
import hashlib
import json
import os
from pathlib import Path
import platform
import time

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np
import sklearn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from threadpoolctl import threadpool_limits


def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    p = np.exp(z)
    return p / p.sum(axis=-1, keepdims=True)


def cross_entropy(logits, labels):
    shifted = logits - logits.max(axis=1, keepdims=True)
    return float(np.mean(np.log(np.exp(shifted).sum(axis=1))
                         - shifted[np.arange(len(labels)), labels]))


class Linear:
    """Affine 64 -> 10 classifier; no hidden layer or activation."""
    def __init__(self, seed=42, d=64, classes=10):
        rng = np.random.default_rng(seed)
        self.p = {"W": rng.normal(0, 0.01, (d, classes)),
                  "b": np.zeros(classes)}

    def forward(self, x):
        return x @ self.p["W"] + self.p["b"], None

    def loss_grad(self, x, y, alpha=0.0):
        logits, _ = self.forward(x)
        dz = softmax(logits)
        dz[np.arange(len(y)), y] -= 1
        dz /= len(y)
        ce = cross_entropy(logits, y)
        return ce, {"W": x.T @ dz, "b": dz.sum(axis=0)}, {"ce": ce, "balance": 0.0}


class MoE:
    """Independent copied linear experts with genuine sparse top-k dispatch.

    Only the selected experts process a sample. No capacity limit or dropping.
    Mix logits using softmax over selected router scores, then classify.
    """
    def __init__(self, dense, seed=1042, experts=4, top_k=2):
        if not 1 < top_k <= experts:
            raise ValueError("Use at least two selected experts for task router gradients.")
        d, c = dense.p["W"].shape
        self.experts, self.top_k = experts, top_k
        rng = np.random.default_rng(seed)
        self.p = {"W": np.repeat(dense.p["W"][None], experts, axis=0),
                  "b": np.repeat(dense.p["b"][None], experts, axis=0),
                  "R": rng.normal(0, 0.05, (d, experts)),
                  "r": np.zeros(experts)}

    @classmethod
    def from_scratch(cls, seed, d=64, classes=10, experts=4, top_k=2):
        """Same architecture, with independently initialized untrained experts."""
        model = cls(Linear(seed, d, classes), seed + 1000, experts, top_k)
        rng = np.random.default_rng(seed + 3000)
        model.p["W"] = rng.normal(0, 0.01, (experts, d, classes))
        return model

    def forward(self, x):
        scores = x @ self.p["R"] + self.p["r"]
        selected = np.argsort(-scores, axis=1, kind="stable")[:, :self.top_k]
        weights = softmax(np.take_along_axis(scores, selected, axis=1))
        expert_logits = np.zeros((len(x), self.top_k, self.p["b"].shape[1]))
        for e in range(self.experts):
            rows, slots = np.where(selected == e)
            expert_logits[rows, slots] = x[rows] @ self.p["W"][e] + self.p["b"][e]
        logits = (weights[:, :, None] * expert_logits).sum(axis=1)
        cache = (selected, weights, expert_logits, softmax(scores))
        return logits, cache

    def loss_grad(self, x, y, alpha=0.01):
        logits, (selected, weights, expert_logits, full_probs) = self.forward(x)
        n = len(y)
        dz = softmax(logits)
        dz[np.arange(n), y] -= 1
        dz /= n
        grads = {name: np.zeros_like(value) for name, value in self.p.items()}
        for e in range(self.experts):
            rows, slots = np.where(selected == e)
            de = dz[rows] * weights[rows, slots, None]
            grads["W"][e] = x[rows].T @ de
            grads["b"][e] = de.sum(axis=0)
        # Within a fixed top-k set, differentiate its normalized softmax weights.
        dweights = (expert_logits * dz[:, None, :]).sum(axis=2)
        dselected_scores = weights * (dweights - (weights * dweights).sum(axis=1, keepdims=True))
        dscores = np.zeros_like(full_probs)
        np.add.at(dscores, (np.arange(n)[:, None], selected), dselected_scores)
        # Switch-style auxiliary balancing; hard assignments are stop-gradient.
        f = np.bincount(selected.ravel(), minlength=self.experts) / (n * self.top_k)
        balance = float(self.experts * np.dot(f, full_probs.mean(axis=0)))
        dscores += alpha * self.experts / n * full_probs * (
            f[None, :] - (full_probs * f).sum(axis=1, keepdims=True))
        grads["R"], grads["r"] = x.T @ dscores, dscores.sum(axis=0)
        ce = cross_entropy(logits, y)
        return ce + alpha * balance, grads, {"ce": ce, "balance": balance}


## 2. Adam, disjoint data splits and resumable checkpoints

Pixels are divided by the known maximum, 16. Split indices are saved. Checkpoints include model weights, Adam moments and the shuffle RNG state, and load without pickle.

In [ ]:


class Adam:
    def __init__(self, params, lr=0.01, router_lr=0.002):
        self.lr, self.router_lr, self.t = lr, router_lr, 0
        self.m = {k: np.zeros_like(v) for k, v in params.items()}
        self.v = {k: np.zeros_like(v) for k, v in params.items()}

    def step(self, params, grads):
        self.t += 1
        for name, grad in grads.items():
            if not np.all(np.isfinite(grad)):
                raise FloatingPointError(f"Nonfinite gradient: {name}")
            self.m[name] = 0.9 * self.m[name] + 0.1 * grad
            self.v[name] = 0.999 * self.v[name] + 0.001 * grad ** 2
            mhat = self.m[name] / (1 - 0.9 ** self.t)
            vhat = self.v[name] / (1 - 0.999 ** self.t)
            lr = self.router_lr if name in ("R", "r") else self.lr
            params[name] -= lr * mhat / (np.sqrt(vhat) + 1e-8)


def get_data(split_seed=2026):
    x, y = load_digits(return_X_y=True)
    x = x.astype(np.float64) / 16.0
    indices = np.arange(len(y))
    train, rest = train_test_split(indices, test_size=0.4, random_state=split_seed, stratify=y)
    val, test = train_test_split(rest, test_size=0.5, random_state=split_seed, stratify=y[rest])
    assert not (set(train) & set(val) or set(train) & set(test) or set(val) & set(test))
    return {"train": (x[train], y[train]), "val": (x[val], y[val]),
            "test": (x[test], y[test]), "indices": {"train": train, "val": val, "test": test},
            "sha256": hashlib.sha256(x.tobytes() + y.astype(np.int64).tobytes()).hexdigest()}


def evaluate(model, data):
    x, y = data
    z, _ = model.forward(x)
    return {"ce": cross_entropy(z, y), "accuracy": float(np.mean(z.argmax(axis=1) == y))}


def save_checkpoint(path, model, optimizer=None, rng=None, metadata=None):
    meta = dict(metadata or {})
    meta.update({"model": type(model).__name__, "experts": getattr(model, "experts", 1),
                 "top_k": getattr(model, "top_k", 1)})
    arrays = {"param_" + k: v for k, v in model.p.items()}
    if optimizer is not None:
        meta["adam"] = {"t": optimizer.t, "lr": optimizer.lr, "router_lr": optimizer.router_lr}
        arrays.update({"adam_m_" + k: v for k, v in optimizer.m.items()})
        arrays.update({"adam_v_" + k: v for k, v in optimizer.v.items()})
    if rng is not None:
        meta["rng"] = rng.bit_generator.state
    arrays["metadata_json"] = np.array(json.dumps(meta))
    np.savez_compressed(path, **arrays)


def load_checkpoint(path):
    with np.load(path, allow_pickle=False) as z:
        meta = json.loads(str(z["metadata_json"]))
        model = Linear(d=z["param_W"].shape[-2], classes=z["param_W"].shape[-1])
        if meta["model"] == "MoE":
            model = MoE(model, experts=meta["experts"], top_k=meta["top_k"])
        model.p = {k: z["param_" + k].copy() for k in model.p}
        optimizer = None
        if "adam" in meta:
            cfg = meta["adam"]
            optimizer = Adam(model.p, cfg["lr"], cfg["router_lr"])
            optimizer.t = cfg["t"]
            optimizer.m = {k: z["adam_m_" + k].copy() for k in model.p}
            optimizer.v = {k: z["adam_v_" + k].copy() for k in model.p}
    rng = np.random.default_rng()
    if "rng" in meta:
        rng.bit_generator.state = meta["rng"]
    return model, optimizer, rng, meta


## 3. Training protocol and comparisons

- Pretrain a linear model for 20 epochs, then fork the checkpoint.
- Continue the linear and upcycled MoE branches for 80 epochs.
- The scratch MoE trains for the same total number of epochs; Adam resets at
  the same boundary. All branches share their corresponding batch orders.
- The projection-budget linear control receives enough extra epochs to match
  the MoE's cumulative affine projection MACs, including linear pretraining.
- Remove the balancing loss in a fifth branch while preserving its starting
  checkpoint and batches. No result is removed if an ablation wins.

The MAC budget excludes backward, optimizer, sort, softmax and mixing costs.
It is not a complete FLOP or wall-clock match. Test labels are used only for
fixed-final-checkpoint evaluation; train and validation are logged each epoch.


In [ ]:


def train_phase(model, data, epochs, seed, phase, global_offset, lr, batch_size=64,
                alpha=0.01, router_lr=0.002, verbose=True, optimizer=None, rng=None):
    optimizer = optimizer or Adam(model.p, lr, router_lr)
    rng = rng or np.random.default_rng(seed)
    rows = []
    start = time.perf_counter()
    training_seconds = 0.0
    x, y = data["train"]
    for epoch in range(epochs + 1):
        if epoch:
            training_start = time.perf_counter()
            order = rng.permutation(len(y))
            for ix in range(0, len(y), batch_size):
                batch = order[ix:ix + batch_size]
                _, gradients, _ = model.loss_grad(x[batch], y[batch], alpha)
                optimizer.step(model.p, gradients)
            training_seconds += time.perf_counter() - training_start
        train, val = evaluate(model, data["train"]), evaluate(model, data["val"])
        row = {"phase": phase, "epoch": epoch, "global_epoch": epoch + global_offset,
               "train_ce": train["ce"], "val_ce": val["ce"],
               "train_accuracy": train["accuracy"], "val_accuracy": val["accuracy"],
               "elapsed_seconds": time.perf_counter() - start,
               "training_seconds": training_seconds}
        if isinstance(model, MoE):
            _, cache = model.forward(x)
            selected, weights, _, probs = cache
            f = np.bincount(selected.ravel(), minlength=model.experts) / selected.size
            row["balance_loss"] = float(model.experts * np.dot(f, probs.mean(axis=0)))
            row["objective"] = train["ce"] + alpha * row["balance_loss"]
            row["router_entropy_normalized"] = float(np.mean(
                -np.sum(probs * np.log(np.maximum(probs, 1e-300)), axis=1)) / np.log(model.experts))
            row["selected_gate_entropy_normalized"] = float(np.mean(
                -np.sum(weights * np.log(np.maximum(weights, 1e-300)), axis=1)) / np.log(model.top_k))
            if epoch == 0 or epoch == epochs or epoch % 20 == 0:
                task_grad = model.loss_grad(x, y, 0.0)[1]
                total_grad = model.loss_grad(x, y, alpha)[1]
                row["task_router_grad_l2"] = float(np.linalg.norm(task_grad["R"]))
                row["aux_router_grad_l2"] = float(np.linalg.norm(total_grad["R"] - task_grad["R"]))
            for e in range(model.experts):
                row[f"expert_{e}_assignment_share"] = float(f[e])
        rows.append(row)
        if verbose and (epoch == 0 or epoch == epochs or epoch % 20 == 0):
            print(f"{phase:18s} epoch={epoch:3d} train_CE={train['ce']:.6f} val_CE={val['ce']:.6f}", flush=True)
    return rows, optimizer, rng



def routing_diagnostics(model, data):
    """Validation-only class routing; columns are case-local expert identities."""
    x, y = data["val"]
    _, (selected, weights, _, probs) = model.forward(x)
    class_counts = np.array([np.bincount(selected[y == c].ravel(),
        minlength=model.experts) for c in range(model.p["b"].shape[1])])
    return {"split": "validation", "class_labels": list(range(len(class_counts))),
        "class_counts": class_counts.tolist(),
        "class_assignment_shares": (class_counts / class_counts.sum(axis=1, keepdims=True)).tolist(),
        "class_sample_counts": np.bincount(y, minlength=len(class_counts)).tolist(),
        "router_entropy_normalized": float(np.mean(-np.sum(probs * np.log(np.maximum(probs, 1e-300)), axis=1)) / np.log(model.experts)),
        "selected_gate_entropy_normalized": float(np.mean(-np.sum(weights * np.log(np.maximum(weights, 1e-300)), axis=1)) / np.log(model.top_k))}


def stats(values):
    values = np.asarray(values, dtype=float)
    return {"mean": float(values.mean()),
            "std": float(values.std(ddof=1)) if len(values) > 1 else 0.0,
            "n": len(values)}


def aggregate_summaries(summaries):
    """Descriptive SD, never an iid claim about overlapping data splits."""
    numeric = [k for k, v in summaries[0].items()
               if isinstance(v, (int, float)) and k not in ("seed", "split_seed")]
    result = {k: stats([s[k] for s in summaries]) for k in numeric}
    for stage in ("linear_pretrain_test", "linear_final_test", "moe_final_test"):
        result[stage] = {m: stats([s[stage][m] for s in summaries]) for m in ("ce", "accuracy")}
    result["branches"] = {name: {split: {m: stats([s["branches"][name][split][m] for s in summaries])
        for m in ("ce", "accuracy")} for split in ("train", "val", "test")}
        for name in summaries[0]["branches"]}
    comparisons = {}
    for baseline in ("linear", "linear_mac_matched", "scratch_moe", "moe_no_balance"):
        comparisons[baseline] = {}
        for split, metric in (("val", "ce"), ("test", "ce"), ("test", "accuracy")):
            values = [s["branches"]["moe"][split][metric] - s["branches"][baseline][split][metric]
                      for s in summaries]
            name = f"{split}_{metric}"
            comparisons[baseline][name] = {**stats(values),
                "moe_better": sum(v < -1e-12 if metric == "ce" else v > 1e-12 for v in values),
                "ties": sum(abs(v) <= 1e-12 for v in values),
                "per_split_mean_delta": {str(seed): float(np.mean([v for s, v in zip(summaries, values)
                    if s["split_seed"] == seed])) for seed in sorted({s["split_seed"] for s in summaries})}}
    result["paired_comparisons"] = comparisons
    result["per_split"] = {str(seed): {name: {split: stats([s["branches"][name][split]["ce"]
        for s in summaries if s["split_seed"] == seed]) for split in ("train", "val", "test")}
        for name in summaries[0]["branches"]} for seed in sorted({s["split_seed"] for s in summaries})}
    return result


def run_seed(seed, data, out, pre_epochs=20, post_epochs=80, verbose=True,
             split_seed=2026, experts=4, top_k=2, batch_size=64, alpha=0.01):
    seed_dir = out / f"seed_{seed}"
    seed_dir.mkdir(parents=True, exist_ok=True)
    common = {"batch_size": batch_size, "router_lr": 0.002, "verbose": verbose}
    linear = Linear(seed)
    pre, opt, rng = train_phase(linear, data, pre_epochs, seed + 100,
                               "linear_pretrain", 0, 0.01, **common)
    meta = {"seed": seed, "split_seed": split_seed, "epoch": pre_epochs}
    save_checkpoint(seed_dir / "linear_pretrained.npz", linear, opt, rng, meta)
    converted = MoE(linear, seed + 1000, experts, top_k)
    all_x = np.concatenate([data[s][0] for s in ("train", "val", "test")])
    delta = float(np.max(np.abs(linear.forward(all_x)[0] - converted.forward(all_x)[0])))
    if delta > 1e-12:
        raise AssertionError(f"Conversion changed logits: {delta}")
    save_checkpoint(seed_dir / "moe_converted.npz", converted, metadata=meta)
    d, c = linear.p["W"].shape
    linear_macs, moe_macs = d * c, d * experts + top_k * d * c
    # Matches affine projection MACs; excludes gates, mixing, sorting, backward,
    # and optimizer work. The rounded-up dense budget is explicitly recorded.
    mac_epochs = int(np.ceil(post_epochs * moe_macs / linear_macs))
    models = {"linear": copy.deepcopy(linear), "moe": copy.deepcopy(converted),
              "linear_mac_matched": copy.deepcopy(linear),
              "moe_no_balance": copy.deepcopy(converted),
              "scratch_moe": MoE.from_scratch(seed, d, c, experts, top_k)}
    scratch_pre, _, _ = train_phase(models["scratch_moe"], data, pre_epochs,
        seed + 100, "scratch_pretrain", 0, 0.01, alpha=alpha, **common)
    phase_names = {"linear": "linear_continue", "moe": "moe_continue",
        "linear_mac_matched": "linear_mac_matched", "moe_no_balance": "moe_no_balance",
        "scratch_moe": "scratch_continue"}
    histories, phase_rows, branches = pre + scratch_pre, {}, {}
    for name, model in models.items():
        epochs = mac_epochs if name == "linear_mac_matched" else post_epochs
        branch_alpha = 0.0 if name == "moe_no_balance" else alpha
        rows, optimizer, batch_rng = train_phase(model, data, epochs, seed + 200,
            phase_names[name], pre_epochs, 0.005, alpha=branch_alpha, **common)
        histories += rows
        phase_rows[name] = rows
        save_checkpoint(seed_dir / f"{name}_final.npz", model, optimizer, batch_rng,
            {"seed": seed, "split_seed": split_seed, "epoch": pre_epochs + epochs,
             "continuation_epochs": epochs, "balance_alpha": branch_alpha})
        pre_macs = moe_macs if name == "scratch_moe" else linear_macs
        post_macs = moe_macs if isinstance(model, MoE) else linear_macs
        pre_seconds = scratch_pre[-1]["training_seconds"] if name == "scratch_moe" else pre[-1]["training_seconds"]
        branches[name] = {"train": evaluate(model, data["train"]),
            "val": evaluate(model, data["val"]), "test": evaluate(model, data["test"]),
            "epochs": pre_epochs + epochs, "continuation_epochs": epochs,
            "forward_projection_macs": len(data["train"][1]) * (pre_epochs * pre_macs + epochs * post_macs),
            "training_seconds": pre_seconds + rows[-1]["training_seconds"]}
    for row in histories:
        row.update({"seed": seed, "split_seed": split_seed})
    columns = list(dict.fromkeys(k for row in histories for k in row))
    with (seed_dir / "history.csv").open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=columns)
        writer.writeheader()
        writer.writerows(histories)
    moe = models["moe"]
    summary = {"seed": seed, "split_seed": split_seed,
        "linear_params": sum(v.size for v in linear.p.values()),
        "moe_total_params": sum(v.size for v in moe.p.values()),
        "moe_active_params_per_sample": top_k * (moe.p["W"][0].size + moe.p["b"][0].size) + moe.p["R"].size + moe.p["r"].size,
        "linear_projection_macs_per_sample": linear_macs,
        "moe_projection_macs_per_sample": moe_macs,
        "conversion_max_abs_logit_difference": delta,
        "initial_train_ce": pre[0]["train_ce"], "initial_val_ce": pre[0]["val_ce"],
        "linear_pretrain_train_ce": pre[-1]["train_ce"], "linear_pretrain_val_ce": pre[-1]["val_ce"],
        "linear_final_train_ce": branches["linear"]["train"]["ce"],
        "linear_final_val_ce": branches["linear"]["val"]["ce"],
        "moe_initial_train_ce": phase_rows["moe"][0]["train_ce"],
        "moe_initial_val_ce": phase_rows["moe"][0]["val_ce"],
        "moe_final_train_ce": branches["moe"]["train"]["ce"],
        "moe_final_val_ce": branches["moe"]["val"]["ce"],
        "linear_pretrain_test": evaluate(linear, data["test"]),
        "linear_final_test": branches["linear"]["test"], "moe_final_test": branches["moe"]["test"],
        "router_weight_change_l2": float(np.linalg.norm(moe.p["R"] - converted.p["R"])),
        "expert_weight_change_l2": [float(np.linalg.norm(moe.p["W"][e] - converted.p["W"][e])) for e in range(experts)],
        "mean_pairwise_expert_weight_distance": float(np.mean([
            np.linalg.norm(moe.p["W"][a] - moe.p["W"][b]) for a in range(experts) for b in range(a + 1, experts)])),
        "final_training_assignment_share": [phase_rows["moe"][-1][f"expert_{e}_assignment_share"] for e in range(experts)],
        "branches": branches}
    for split in ("train", "val"):
        assert summary[f"linear_pretrain_{split}_ce"] < summary[f"initial_{split}_ce"]
        assert summary[f"linear_final_{split}_ce"] < summary[f"linear_pretrain_{split}_ce"]
        assert summary[f"moe_final_{split}_ce"] < summary[f"moe_initial_{split}_ce"]
    assert summary["router_weight_change_l2"] > 0
    assert min(summary["expert_weight_change_l2"]) > 0
    diagnostics = {name: routing_diagnostics(models[name], data)
        for name in ("moe", "scratch_moe", "moe_no_balance")}
    (seed_dir / "routing.json").write_text(json.dumps(diagnostics, indent=2) + "\n")
    (seed_dir / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
    return summary, histories


def run_experiment(out="results", seeds=(42, 43, 44, 45, 46), pre_epochs=20,
                   post_epochs=80, verbose=True, split_seeds=(2026, 2027, 2028),
                   experts=4, top_k=2, batch_size=64, alpha=0.01):
    if not seeds or not split_seeds or len(set(seeds)) != len(seeds) or len(set(split_seeds)) != len(split_seeds):
        raise ValueError("Provide nonempty, unique initialization seeds and split seeds.")
    if min(pre_epochs, post_epochs, batch_size) < 1 or alpha < 0 or not 1 < top_k <= experts:
        raise ValueError("Positive epochs/batch size, nonnegative alpha, and 1 < top_k <= experts required.")
    out = Path(out)
    out.mkdir(parents=True, exist_ok=True)
    data = get_data(split_seeds[0])
    config = {"schema_version": 2, "seeds": list(seeds), "split_seeds": list(split_seeds),
        "pretrain_epochs": pre_epochs, "continuation_epochs": post_epochs, "batch_size": batch_size,
        "pretrain_lr": 0.01, "continuation_lr": 0.005, "router_lr": 0.002,
        "balance_alpha": alpha, "experts": experts, "top_k": top_k,
        "dtype": "float64", "device": "CPU", "blas_threads": 1,
        "split_counts": {s: len(data[s][1]) for s in ("train", "val", "test")},
        "dataset_sha256": data["sha256"], "python": platform.python_version(),
        "numpy": np.__version__, "scikit_learn": sklearn.__version__,
        "hardware": platform.platform(), "cpu_count": os.cpu_count(),
        "note": "Fixed final epochs; no checkpoint selection or test tuning. Adam resets after pretraining in every branch. Projection MAC budget excludes backward, optimizer, gating, sorting and mixing. Splits overlap; variation is descriptive."}
    (out / "config.json").write_text(json.dumps(config, indent=2) + "\n")
    summaries, histories = [], []
    with threadpool_limits(limits=1):
        for split_seed in split_seeds:
            data = get_data(split_seed)
            split_dir = out / f"split_{split_seed}"
            split_dir.mkdir(exist_ok=True)
            np.savez_compressed(split_dir / "split_indices.npz", **data["indices"])
            for seed in seeds:
                if verbose:
                    print(f"\n=== Split {split_seed} / seed {seed} ===", flush=True)
                summary, history = run_seed(seed, data, split_dir, pre_epochs, post_epochs,
                    verbose, split_seed, experts, top_k, batch_size, alpha)
                summaries.append(summary)
                histories.extend(history)
    (out / "summary.json").write_text(json.dumps(summaries, indent=2) + "\n")
    (out / "aggregate.json").write_text(json.dumps(aggregate_summaries(summaries), indent=2) + "\n")
    comparison = [{"split_seed": s["split_seed"], "seed": s["seed"], "model": name,
        "epochs": b["epochs"], "train_ce": b["train"]["ce"], "val_ce": b["val"]["ce"],
        "test_ce": b["test"]["ce"], "test_accuracy": b["test"]["accuracy"],
        "forward_projection_macs": b["forward_projection_macs"]}
        for s in summaries for name, b in s["branches"].items()]
    with (out / "comparison.csv").open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(comparison[0]))
        writer.writeheader()
        writer.writerows(comparison)
    print("\nAll runs passed loss-decrease and conversion assertions.", flush=True)
    return summaries, histories




## 4. Numerical verification

Check all handwritten gradients by finite differences, the sparse dispatch reference, independent expert storage, exact resumption and tolerance behavior.

In [ ]:
"""Compare numerical evidence with explicit tolerances, excluding run timings."""
from numbers import Real
import numpy as np

TIMING_KEYS = frozenset({"training_seconds", "elapsed_seconds"})


def compare_evidence(actual, expected, *, rtol=1e-6, atol=1e-8, path="root"):
    if isinstance(expected, dict):
        assert isinstance(actual, dict) and actual.keys() == expected.keys(), f"{path}: dictionary keys differ"
        for key in expected.keys() - TIMING_KEYS:
            compare_evidence(actual[key], expected[key], rtol=rtol, atol=atol, path=f"{path}/{key}")
    elif isinstance(expected, (list, tuple)):
        assert isinstance(actual, (list, tuple)) and len(actual) == len(expected), f"{path}: length differs"
        for i, (new, old) in enumerate(zip(actual, expected)):
            compare_evidence(new, old, rtol=rtol, atol=atol, path=f"{path}/{i}")
    elif isinstance(expected, Real) and not isinstance(expected, (bool, int)):
        np.testing.assert_allclose(actual, expected, rtol=rtol, atol=atol, err_msg=path)
    else:
        assert actual == expected, f"{path}: {actual!r} != {expected!r}"


In [ ]:
"""Numerical checks for handwritten gradients, conversion, and resumption."""
import copy
import json
from pathlib import Path
import tempfile
import numpy as np


def check_gradients(model, x, y, alpha):
    _, grads, _ = model.loss_grad(x, y, alpha)
    worst = 0.0
    count = 0
    for name, parameter in model.p.items():
        for index in np.ndindex(parameter.shape):
            old = parameter[index]
            parameter[index] = old + 1e-6
            plus = model.loss_grad(x, y, alpha)[0]
            parameter[index] = old - 1e-6
            minus = model.loss_grad(x, y, alpha)[0]
            parameter[index] = old
            numeric = (plus - minus) / 2e-6
            analytic = grads[name][index]
            error = abs(numeric - analytic) / max(1.0, abs(numeric), abs(analytic))
            worst = max(worst, error)
            count += 1
    assert worst < 1e-7, worst
    return {"parameters_checked": count, "max_scaled_error": float(worst)}


def verify():
    rng = np.random.default_rng(90210)
    x, y = rng.normal(size=(11, 4)), rng.integers(0, 3, 11)
    linear = Linear(8, d=4, classes=3)
    moe = MoE(linear, 99)
    initial_task_gradient = np.linalg.norm(moe.loss_grad(x, y, 0.0)[1]["R"])
    assert initial_task_gradient < 1e-14
    scratch = MoE.from_scratch(8, d=4, classes=3)
    assert not np.array_equal(scratch.p["W"][0], scratch.p["W"][1])
    compare_evidence({"loss": 0.123456789 + 1e-9, "training_seconds": 3.0},
                     {"loss": 0.123456789, "training_seconds": 1.0})
    try:
        compare_evidence({"loss": 0.13}, {"loss": 0.12})
    except AssertionError:
        pass
    else:
        raise AssertionError("Numerical comparison accepted a material loss change.")
    delta = float(np.max(np.abs(linear.forward(x)[0] - moe.forward(x)[0])))
    assert delta < 1e-12
    assert not np.shares_memory(linear.p["W"], moe.p["W"])
    for a in range(4):
        for b in range(a + 1, 4):
            assert not np.shares_memory(moe.p["W"][a], moe.p["W"][b])
    # Break expert symmetry for a meaningful router task-gradient check.
    moe.p["W"] += rng.normal(0, 0.1, moe.p["W"].shape)
    moe.p["b"] += rng.normal(0, 0.1, moe.p["b"].shape)
    checks = {"linear_gradient": check_gradients(linear, x, y, 0.0),
              "moe_task_gradient": check_gradients(moe, x, y, 0.0),
              "moe_task_plus_balance_gradient": check_gradients(moe, x, y, 0.01),
              "conversion_max_abs_error": delta}
    logits, (selected, weights, _, _) = moe.forward(x)
    assert np.allclose(weights.sum(axis=1), 1)
    assert np.all(selected[:, 0] != selected[:, 1])
    assert selected.size == len(x) * 2
    # Independent dense reference is used ONLY to verify sparse dispatch.
    reference = np.zeros_like(logits)
    for i in range(len(x)):
        for slot, expert in enumerate(selected[i]):
            reference[i] += weights[i, slot] * (x[i] @ moe.p["W"][expert] + moe.p["b"][expert])
    assert np.allclose(logits, reference, atol=1e-14)
    optimizer = Adam(moe.p)
    optimizer.step(moe.p, moe.loss_grad(x, y)[1])
    with tempfile.TemporaryDirectory() as folder:
        path = Path(folder) / "resume.npz"
        save_checkpoint(path, moe, optimizer, rng)
        restored, restored_opt, restored_rng, _ = load_checkpoint(path)
        assert np.array_equal(rng.permutation(11), restored_rng.permutation(11))
        optimizer.step(moe.p, moe.loss_grad(x, y)[1])
        restored_opt.step(restored.p, restored.loss_grad(x, y)[1])
        assert all(np.array_equal(moe.p[k], restored.p[k]) for k in moe.p)
    checks.update({"sparse_dispatch_reference": "passed", "normalized_top2_weights": "passed",
                   "independent_expert_storage": "passed", "checkpoint_exact_next_step": "passed"})
    checks.update({"copied_experts_initial_task_router_gradient_l2": float(initial_task_gradient),
                   "scratch_experts_independently_initialized": "passed",
                   "numerical_tolerance_accepts_roundoff_rejects_material_change": "passed"})
    return checks




In [ ]:
checks = verify()
print(json.dumps(checks, indent=2))
Path("results").mkdir(exist_ok=True)
Path("results/verification.json").write_text(json.dumps(checks, indent=2) + "\n")


## 5. Execute all 15 paired cases

The final epoch is reported for every branch. No best checkpoint or best seed is selected.

In [ ]:
summaries, histories = run_experiment("results", verbose=False, seeds=(42, 43, 44, 45, 46), split_seeds=(2026, 2027, 2028), pre_epochs=20, post_epochs=80, experts=4, top_k=2, batch_size=64, alpha=0.01)
for s in summaries:
    print(f"Split {s['split_seed']}, seed {s['seed']}: "
          f"MoE train CE {s['moe_initial_train_ce']:.6f} → {s['moe_final_train_ce']:.6f}; "
          f"validation {s['moe_initial_val_ce']:.6f} → {s['moe_final_val_ce']:.6f}")


## 6. Read the actual aggregate and paired comparisons

Cross-entropy is in nats and excludes the auxiliary loss. SD is a sample standard deviation, not a confidence interval. Paired deltas retain each case's seed and split.

In [ ]:
aggregate = json.loads(Path("results/aggregate.json").read_text())
print(f"{'Branch':<24} {'Train CE':>10} {'Val CE':>10} {'Test CE':>10} {'Test accuracy':>15}")
for name, branch in aggregate["branches"].items():
    print(f"{name:<24} {branch['train']['ce']['mean']:10.6f} {branch['val']['ce']['mean']:10.6f} "
          f"{branch['test']['ce']['mean']:10.6f} {100*branch['test']['accuracy']['mean']:14.3f}%")
print("\nPaired MoE minus baseline (negative validation CE delta favors MoE):")
for name, values in aggregate["paired_comparisons"].items():
    v = values["val_ce"]
    print(name, "mean delta:", round(v["mean"], 6), "MoE wins:", v["moe_better"], "of", v["n"],
          "per-split mean deltas:", v["per_split_mean_delta"])
print("\nMaximum conversion logit difference:", max(s["conversion_max_abs_logit_difference"] for s in summaries))


## 7. Inspect losses, baselines, projection budgets and the router

Plots use the saved raw CSV/JSON data without smoothing. Router heatmaps show validation labels in the first configured case. Expert IDs are local to each run; a routing pattern is not a semantic expert label.

In [ ]:
"""Plot measured CSV/JSON evidence. No smoothing; singleton SD is zero."""
import csv
import json
from pathlib import Path
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

COLORS = {"pre": "#5968d8", "linear": "#bb7633", "moe": "#008b82",
          "linear_mac_matched": "#ce995c", "scratch_moe": "#775bc4", "moe_no_balance": "#bc5273"}
LABELS = {"linear": "Linear · 100 epochs", "moe": "Upcycled MoE · 100",
          "linear_mac_matched": "Linear · projection budget", "scratch_moe": "Scratch MoE · 100",
          "moe_no_balance": "Upcycled MoE · no balance"}


def sample_sd(values, axis=0):
    a = np.asarray(values)
    return a.std(axis=axis, ddof=1) if a.shape[axis] > 1 else np.zeros_like(a.mean(axis=axis))


def plot_results(folder="results"):
    folder = Path(folder)
    summaries = json.loads((folder / "summary.json").read_text())
    cfg = json.loads((folder / "config.json").read_text())
    histories = []
    for s in summaries:
        with (folder / f"split_{s['split_seed']}" / f"seed_{s['seed']}" / "history.csv").open() as f:
            histories.append(list(csv.DictReader(f)))
    n, e = len(summaries), cfg["experts"]
    caption = f"{len(cfg['seeds'])} initialization seed(s) × {len(cfg['split_seeds'])} split(s) = {n} paired run(s)"
    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10,
        "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold",
        "text.color": "#192936", "axes.edgecolor": "#bac5ce", "savefig.facecolor": "white"})
    paths = []

    def save(fig, name):
        fig.savefig(folder / (name + ".png"), dpi=170)
        fig.savefig(folder / (name + ".svg"))
        paths.append(folder / (name + ".png"))
        plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.0), layout="constrained")
    for ax, key, title in zip(axes, ("train_ce", "val_ce"), ("Training loss", "Validation loss")):
        for phase, color, label in (("linear_pretrain", COLORS["pre"], "Linear pretraining"),
                ("linear_continue", COLORS["linear"], "Linear continuation"),
                ("moe_continue", COLORS["moe"], "Upcycled MoE continuation")):
            curves = [[r for r in h if r["phase"] == phase] for h in histories]
            xs = [int(r["global_epoch"]) for r in curves[0]]
            ys = np.array([[float(r[key]) for r in rows] for rows in curves])
            mean, sd = ys.mean(axis=0), sample_sd(ys)
            ax.plot(xs, mean, color=color, linewidth=2.3, label=label)
            ax.fill_between(xs, np.maximum(mean - sd, 1e-8), mean + sd, color=color, alpha=.13)
        ax.axvline(cfg["pretrain_epochs"], color="#78838f", linestyle="--", linewidth=1)
        ax.text(cfg["pretrain_epochs"] + 2, .93, "Convert", transform=ax.get_xaxis_transform(), fontsize=9)
        ax.set(title=title, xlabel="Epoch within each branch", ylabel="Cross-entropy (nats)", yscale="log")
        ax.grid(axis="y", alpha=.15)
    axes[0].legend(frameon=False, fontsize=9)
    fig.suptitle("A trained linear model becomes a trainable MoE", fontsize=18, weight="bold")
    fig.supxlabel(caption + " · mean ± sample SD; descriptive variation", fontsize=9)
    save(fig, "loss_curves")

    names = list(LABELS)
    fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.7), layout="constrained")
    for ax, split, metric, title in ((axes[0], "val", "ce", "Final validation cross-entropy"),
                                   (axes[1], "test", "accuracy", "Final test accuracy")):
        vals = np.array([[s["branches"][name][split][metric] for s in summaries] for name in names])
        scale = 100 if metric == "accuracy" else 1
        ax.barh(np.arange(len(names)), vals.mean(axis=1) * scale,
                xerr=sample_sd(vals, axis=1) * scale, color=[COLORS[name] for name in names],
                alpha=.9, capsize=3)
        ax.set(yticks=np.arange(len(names)), yticklabels=[LABELS[name] for name in names], title=title,
               xlabel="Accuracy (%)" if metric == "accuracy" else "Cross-entropy (nats)")
        ax.invert_yaxis()
        if metric == "accuracy":
            ax.set_xlim(max(0, 100 * vals.min() - 2), 100)
            ax.text(.02, .02, "Accuracy axis is truncated", transform=ax.transAxes, fontsize=8)
        ax.grid(axis="x", alpha=.15)
    fig.suptitle("Stronger baselines at fixed final checkpoints", fontsize=17, weight="bold")
    fig.supxlabel(caption + " · bars: mean ± sample SD · epoch counts differ for budget control", fontsize=9)
    # Epoch labels are derived from config, not assumed for custom runs.
    display_names = {"linear": "Linear", "moe": "Upcycled MoE",
        "linear_mac_matched": "Linear (projection budget)", "scratch_moe": "Scratch MoE",
        "moe_no_balance": "Upcycled MoE, no balance"}
    labels = [f"{display_names[name]} · {summaries[0]['branches'][name]['epochs']} epochs" for name in names]
    for ax in axes:
        ax.set_yticklabels(labels)
    save(fig, "baseline_comparison")

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.9), layout="constrained")
    min_shares = [min(s["final_training_assignment_share"]) for s in summaries]
    axes[0].bar(np.arange(n), min_shares, color=COLORS["moe"], alpha=.85)
    axes[0].axhline(1 / e, color="#7c8690", ls="--", label="Uniform share")
    axes[0].set(xticks=np.arange(n), xticklabels=[f"{s['split_seed']}/{s['seed']}" for s in summaries],
        ylabel="Smallest expert assignment share", title="No expert is unused in the final run")
    axes[0].tick_params(axis="x", labelrotation=55, labelsize=8)
    axes[0].set_ylim(0, max(1 / e, max(min_shares)) * 1.25)
    axes[0].legend(frameon=False, fontsize=8)
    case = summaries[0]
    rows = [r for r in histories[0] if r["phase"] == "moe_continue"]
    for expert in range(e):
        axes[1].plot([int(r["epoch"]) for r in rows],
                     [float(r[f"expert_{expert}_assignment_share"]) for r in rows], label=f"Expert {expert}")
    axes[1].set(title=f"First configured case: {case['split_seed']}/{case['seed']}",
                xlabel="Epochs after conversion", ylabel="Share of selected assignments")
    axes[1].legend(ncol=2, frameon=False, fontsize=8)
    for ax in axes:
        ax.grid(axis="y", alpha=.15)
    fig.suptitle("All experts participate in training", fontsize=18, weight="bold")
    fig.supxlabel("Expert identities are local to each run; curves show one predetermined case", fontsize=9)
    save(fig, "expert_routing")

    routing = json.loads((folder / f"split_{case['split_seed']}" / f"seed_{case['seed']}" / "routing.json").read_text())
    fig, axes = plt.subplots(2, 2, figsize=(12.5, 9), layout="constrained")
    for ax, name, title in ((axes[0, 0], "moe", "Validation class routing: balanced MoE"),
                            (axes[0, 1], "moe_no_balance", "Validation class routing: no balance")):
        shares = np.array(routing[name]["class_assignment_shares"])
        im = ax.imshow(shares, vmin=0, vmax=1 / cfg["top_k"], cmap="YlGnBu", aspect="auto")
        ax.set(title=title, xlabel="Expert ID within this case", ylabel="Digit class",
               xticks=range(e), yticks=range(len(shares)))
        for cls in range(len(shares)):
            for expert in range(e):
                ax.text(expert, cls, f"{shares[cls, expert]:.2f}", ha="center", va="center", fontsize=8,
                        color="white" if shares[cls, expert] > .65 / cfg["top_k"] else "#18303a")
        fig.colorbar(im, ax=ax, shrink=.85, label="Share of class assignments")
    entropy_ax, grad_ax = axes[1]
    for phase, label, color in (("moe_continue", "Balanced", COLORS["moe"]),
            ("moe_no_balance", "No balance", COLORS["moe_no_balance"])):
        r = [row for row in histories[0] if row["phase"] == phase]
        xs = [int(row["epoch"]) for row in r]
        entropy_ax.plot(xs, [float(row["router_entropy_normalized"]) for row in r], color=color,
            label=label + " · all experts")
        entropy_ax.plot(xs, [float(row["selected_gate_entropy_normalized"]) for row in r],
            color=color, ls="--", label=label + " · selected gates")
    entropy_ax.set(title="Router uncertainty during training", xlabel="Epochs after conversion",
        ylabel="Entropy / log(number of choices)", ylim=(0, 1.03))
    entropy_ax.legend(frameon=False, fontsize=8)
    probes = [r for r in rows if r.get("task_router_grad_l2")]
    for key, label in (("task_router_grad_l2", "Task gradient"), ("aux_router_grad_l2", "Balance gradient")):
        grad_ax.plot([int(r["epoch"]) for r in probes], [float(r[key]) for r in probes], "o-", label=label)
    grad_ax.set(title="Router learns after experts diverge", xlabel="Epochs after conversion",
        ylabel="Full-training-set router gradient L2")
    grad_ax.legend(frameon=False, fontsize=8)
    for ax in axes[1]:
        ax.grid(axis="y", alpha=.15)
    fig.suptitle(f"Inspect the router · split {case['split_seed']} / seed {case['seed']}", fontsize=18, weight="bold")
    fig.supxlabel("Heatmaps use validation labels only; class differences are descriptive, not semantic expert labels", fontsize=9)
    save(fig, "router_diagnostics")

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 5), layout="constrained")
    for ax, key, title in zip(axes, ("train_ce", "val_ce"), ("Training", "Validation")):
        for phase, name in (("linear_mac_matched", "linear_mac_matched"), ("moe_continue", "moe")):
            r = [[row for row in h if row["phase"] == phase] for h in histories]
            macs = case["linear_projection_macs_per_sample"] if name == "linear_mac_matched" else case["moe_projection_macs_per_sample"]
            budget = [cfg["pretrain_epochs"] * case["linear_projection_macs_per_sample"] + int(row["epoch"]) * macs for row in r[0]]
            values = np.array([[float(row[key]) for row in run] for run in r])
            ax.plot(np.array(budget) / 1000, values.mean(axis=0), color=COLORS[name], label=name.replace("_", " "))
            sd = sample_sd(values)
            ax.fill_between(np.array(budget) / 1000, np.maximum(values.mean(axis=0) - sd, 1e-8),
                values.mean(axis=0) + sd, color=COLORS[name], alpha=.12)
        ax.set(title=title + " loss vs projection budget", yscale="log", ylabel="Cross-entropy (nats)",
               xlabel="Cumulative affine projection MACs per training example (thousands)")
        ax.grid(axis="y", alpha=.15)
        ax.legend(frameon=False, fontsize=9)
    fig.suptitle("Compare the same forward projection budget", fontsize=17, weight="bold")
    fig.supxlabel("Includes linear pretraining · excludes backward, optimizer, sorting, softmax, and mixing costs", fontsize=9)
    save(fig, "compute_budget")
    return paths




In [ ]:
from IPython.display import display, Image
for path in plot_results("results"):
    display(Image(filename=str(path)))


## 8. Continue training from a saved MoE

Reload the optimizer and shuffle RNG and train for ten additional epochs. This check uses an isolated restored model and does not change the fixed-final-epoch comparison above.

In [ ]:
case = summaries[0]
checkpoint = Path("results") / f"split_{case['split_seed']}" / f"seed_{case['seed']}" / "moe_final.npz"
restored, optimizer, batch_rng, metadata = load_checkpoint(checkpoint)
data = get_data(case["split_seed"])
before = evaluate(restored, data["train"])
np.testing.assert_allclose(evaluate(restored, data["val"])["ce"], case["moe_final_val_ce"], rtol=1e-6, atol=1e-8)
previous_step = optimizer.t
with threadpool_limits(limits=1):
    resumed, optimizer, batch_rng = train_phase(restored, data, epochs=10, seed=0,
        phase="resumed_moe", global_offset=metadata["epoch"], lr=optimizer.lr,
        batch_size=64, alpha=0.01, router_lr=optimizer.router_lr,
        verbose=False, optimizer=optimizer, rng=batch_rng)
after = evaluate(restored, data["train"])
assert after["ce"] < before["ce"]
assert optimizer.t == previous_step + 10 * int(np.ceil(len(data["train"][1]) / 64))
print("Restored model trained further:", before["ce"], "→", after["ce"])
print("Adam steps:", previous_step, "→", optimizer.t)
print("All checks passed.")


## Interpretation and limits

The linear model learns, conversion preserves its predictions, and the
upcycled MoE keeps reducing training and validation loss. Stronger baselines
show that the accuracy advantage varies by case and budget. The balancing
ablation does not consistently improve held-out performance; keep its full
results rather than assuming that balancing must help.

This is a small digit-classification experiment, not a transformer or language
model. Three overlapping data splits are not three independent datasets.
Top-k gradients are local to a fixed selection set. MAC accounting omits
several operations, and CPU timing is implementation-specific. There is no
inference-speed or universal-superiority claim.

References: [Sparse Upcycling](https://arxiv.org/abs/2212.05055),
[Switch Transformers](https://arxiv.org/abs/2101.03961),
[digits dataset](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html).
AI assistance was used to develop and document this experiment; outputs come
from actual executed training and numerical verification.
